In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from pathlib import Path
import pandas as pd
import requests
from ift6758.llm.get_better import (
    extract_text,
    load_function,
    format_test_report,
    build_refine_prompt,
    all_passed,
)
from ift6758.llm.chat import confirm, ask_user
from ift6758.llm.prompts import PROFILES, OG_TACHE_1
from ift6758.llm.qwen_log import create_log, log_call
from ift6758.llm.qwen_utils import (
    load_qwen,
    build_messages,
    generate,
    ask_qwen,
    check_env,
)
from ift6758.llm.rag import (
    load_doc,
    extract_section,
    split_doc,
    load_embedder,
    embed,
    retrieve,
    format_context,
    DOCS_DIR,
)
from ift6758.llm.test_both_code import run_tests, run_season_tests
from ift6758.llm.test_tidy_llm import run_tidy_tests

check_env()  # Demo 2, for reproductibility

Python: 3.11.15
NumPy: 1.26.4 | pandas: 2.3.3
Transformers: 5.17.0
Sentence Transformers: 6.1.0
CUDA GPU: Unavailable — CPU/Apple GPU fallback


In [3]:
# Code adapté de la démo 2
GAME_ID = 2022030411
raw_path = Path(DOCS_DIR.parent / "raw") / f"{GAME_ID}.json"
print("Cached game:", raw_path.resolve())
if not raw_path.exists():
    response = requests.get(
        f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30
    )
    response.raise_for_status()
    downloaded = response.json()
    assert downloaded["id"] == GAME_ID and isinstance(downloaded["plays"], list)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(response.text, encoding="utf-8")
game = json.loads(
    raw_path.read_text(encoding="utf-8")
)  # Convertie de json vers objet python
assert game["id"] == GAME_ID and isinstance(game["plays"], list)
print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

Cached game: /Users/jcforget/Maitrise/Cours/IFT 6758/Projet/Github/NHL-data-analysis/ift6758/data/raw/2022030411.json
FLA at VGK 2023-06-03


In [4]:
# First Try with no conetxt
context = None
results = [None]

Je me suis fait avoir. Le joeur change de nom selon l'event:Un tir a details.shootingPlayerId, mais un but a details.scoringPlayerId (pas de shootingPlayerId). Ensuite, s'il y a un but désert, le but n'a pas de clé goalieInNetId.

In [5]:
# Ici j'ai fait le prompt en réutilisant la structure promposé par Claude et par l'article mais en n'y mettant que les informations que JE connaissais, soit le format voulue (comme Madhi) et les infromations demandé par l'énoncé.
# - The JSON is the play-by-play of one game from the NHL web API:
#  https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play
tidy_task = """
Write a Python function json2tidy_llm(raw_data).
Goal: convert all the events of one NHL game into a tidy pandas DataFrame.

Input:
- raw_data is either a str (path to a JSON file, read as UTF-8) or a dict (already loaded).

Task:
- Keep only shots on goal and goals. Ignore missed shots and blocked shots.
- One row per kept event, uniquely identified by the game ID and the event ID.
- For each event include at least: game ID, event ID, season, game type, period, time in the period,
  the team that took the shot, whether it is a shot or a goal, the coordinates on the ice,
  the shooter name, the goalie name, the shot type, whether the net was empty,
  and, for goals, whether the goal was scored at even strength, shorthanded or on the power play.
- Keep unavailable values as missing. Never replace them with 0 or False.

Output interface (used by automated tests):
- Columns: game_id, event_id, season, game_type, period, time, team_id, event_type,
  x_coord, y_coord, shooter, goalie, shot_type, empty_net, strength.
- event_type is "shot-on-goal" or "goal". game_type is the numeric game type of the game.
- shooter and goalie are full names: "First Last".
- empty_net is True or False.
- strength is "even", "shorthanded" or "power play" for goals, and missing for shots.

Use pandas and json. Include imports and a docstring.
Return only json2tidy_llm and its imports in a Python code block.
"""

In [6]:
model, tokenizer, device = load_qwen()
log_path = create_log()
print("Log :", log_path)


# Code suggested by Claude to embelish presentation
def banner(title):
    print(f"\n{'=' * 15} {title} {'=' * 15}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Log : /Users/jcforget/Maitrise/Cours/IFT 6758/Projet/Github/NHL-data-analysis/logs/chat_2026-10-17_11-03-35.jsonl


In [7]:
MAX_ITER = 5  # SafeGuard
task = tidy_task
task_version = "tidy_v0"  # first version (demo 2 ift6758)
iteration = 0
history = []  # Loop resume, seen at the end

try:
    while True:
        # Code genration section
        banner(f"Step {iteration} : code generation ({task_version})")
        print(task)
        gen_result = ask_qwen(
            model=model,
            tokenizer=tokenizer,
            log_path=log_path,
            prompt=task,
            context=context,
            profile=PROFILES["gen_tidy"],
            extra={
                "iteration": iteration,
                "task_version": task_version,
                "retrieved": results,
            },
        )

        # Extraction and humain in the lopp validation
        try:
            tidy_code = extract_text(gen_result["answer"], tag="code")
        except ValueError as e:
            tidy_code = gen_result["answer"]  # no code generatde
            test_results = {"0. Extraction code": f"ERROR : {e}"}
        else:
            banner("Extracted code")
            print(tidy_code)
            if not confirm("Execute and test code ?"):
                print("Stawp : code ist nicht gut.")
                break
            # Transforme
            try:
                fn = load_function(tidy_code, "json2tidy_llm")
            except Exception as e:  # SyntaxError, import manquant...
                test_results = {"0. Code loading": f"ERROR : {type(e).__name__}: {e}"}
            else:
                test_results = run_tidy_tests(fn)
        report = format_test_report(test_results)
        banner("Test results")
        print(report)
        history.append(
            {
                "iteration": iteration,
                "task_version": task_version,
                "passed": all_passed(test_results),
                "report": report,
            }
        )
        if all_passed(test_results):
            print("\nAll test are Ok !")
            break
        if iteration >= MAX_ITER:
            print(f"\n MAX_ITER ({MAX_ITER}) too many iteration")
            break
        # Prompt refinement
        banner(f"Step {iteration} : prompt refinement")
        refine_prompt = build_refine_prompt(
            task=task,
            code=tidy_code,
            context=context,
            report=report
        )  # Raffine le prompt
        refine_result = ask_qwen(
            model=model,
            tokenizer=tokenizer,
            log_path=log_path,
            prompt=refine_prompt,
            context=None,
            profile=PROFILES["refine"],
            extra={
                "iteration": iteration,
                "task_version": task_version,
                "test_results": test_results,
            },
        )
        try:
            new_task = extract_text(
                answer=refine_result["answer"], tag="new_prompt"
            ).strip()
        except ValueError as e:
            print(f"Stawp!!! : {e}")
            break
        banner("New prompt")
        print(new_task)
        if not confirm("Accept new prompt and generate new code ?"):
            print("Stawp : prompt ist nicht gut.")
            break
        # Netx task
        task = new_task
        iteration += 1
        task_version = f"tidy_v{iteration}"
except KeyboardInterrupt:
    print(f"\n Stopped at step {iteration}.")
pd.DataFrame(history)


=============== Step 0 : code generation (tidy_v0) ===============

Write a Python function json2tidy_llm(raw_data).
Goal: convert all the events of one NHL game into a tidy pandas DataFrame.

Input:
- raw_data is either a str (path to a JSON file, read as UTF-8) or a dict (already loaded).

Task:
- Keep only shots on goal and goals. Ignore missed shots and blocked shots.
- One row per kept event, uniquely identified by the game ID and the event ID.
- For each event include at least: game ID, event ID, season, game type, period, time in the period,
  the team that took the shot, whether it is a shot or a goal, the coordinates on the ice,
  the shooter name, the goalie name, the shot type, whether the net was empty,
  and, for goals, whether the goal was scored at even strength, shorthanded or on the power play.
- Keep unavailable values as missing. Never replace them with 0 or False.

Output interface (used by automated tests):
- Columns: game_id, event_id, season, game_type, period, 

,iteration,task_version,passed,report
0,0,tidy_v0,False,"0/1 tests passed\nTest: 1. Structure, Status: ..."
